## Семинар 9: Случайный лес

### 1. Bias-Variance decomposition

Вспомним, что функцию потерь в задачах регрессии или классификации можно разложить на три компоненты: смещение (bias), дисперсию (variance) и шум (noise). Эти компоненты позволяют описать сложность алгоритма, альтернативно сравнению ошибок на тренировочной и тестовой выборках. Хотя такое разложение можно построить для произвольной функции потерь, наиболее просто (и классически) оно строится для среднеквадратичной функции в задаче регрессии, что мы и рассмотрим ниже. 

Пусть $(X, y)$ – некоторая выборка. Обучим интересующий нас алгоритм на этой выборке и сделаем предсказания на ней. Обозначим предсказания как $\hat{y}$. Тогда 

$$
\mathrm{bias} := \mathbb{E}(\hat{y}) - y.
$$

$$
\mathrm{variance} := \mathbb{E}[\mathbb{E}(\hat{y}) - \hat{y}]^2
$$

$$
\mathrm{noise} := \mathbb{E}[y - \mathbb{E}(y)]^2
$$

Ожидаемую среднеквадратичную ошибку на тренировочной выборке можно разложить как

$$
\mathrm{E}[y - \hat{y}]^2 = \mathrm{bias}^2 + \mathrm{variance} + \mathrm{noise}.
$$

**Техническое замечание:** все математические ожидания в разложении выше берутся по объектам тренировочной выборки, то есть это разложение верно для среднеквадратичной ошибки на тренировочной выборке, которую иногда называют MSE for estimator. Тем не менее, нам интересна и величина ошибки на ненаблюдаемых данных, которую иногда называют MSE for predictor. В этом случае математическое ожидание ошибки следует брать по ненаблюдаемым объектам. Для решения этой проблемы зачастую предполагается, что тренировочная и тестовая выборка имеют одинаковое распределение, и математическое ожидание берётся по всевозможным вариациям тренировочной выборки. Суть разложения при этом не изменится, однако запись его станет более громоздкой. Посмотреть на это можно [здесь](https://towardsdatascience.com/mse-and-bias-variance-decomposition-77449dd2ff55). 

Заметим, что так как на практике мы считаем оценки математических ожиданий и зачастую имеем доступ к тестовой выборке, то проблем с расчётом **оценок** MSE for estimator и MSE for predictor не возникает.

Разберёмся с интерпретацией компонент. 

- $\mathrm{Bias}$ – показывает отклонение среднего ответа алгоритма от ответа идеального алгоритма. $\mathrm{Bias}$ отражает ошибку модели, возникающую из-за простоты модели. Высокое смещение обычно является показателем того, что модель недообучена.


- $\mathrm{Variance}$ – показывает разброс ответов алгоритмов относительно среднего ответа алгоритма. Показывает, насколько сильно небольшие изменения в обучающей выборке скажутся на предсказаниях алгоритма. $\mathrm{Variance}$ отражает ошибку модели, возникающую из-за чрезмерной сложности модели. Высокая дисперсия обычно является показателем того, что модель переобучена.


- $\mathrm{Noise}$ – ошибка идеального классификатора, естественный неустранимый шум в данных. 

Посмотрим наглядно на примере полиномиальной регрессии.

In [ ]:
# Импорт необходимых библиотек
import numpy as np  # для численных операций
import pandas as pd  # для работы с данными 
import matplotlib.pyplot as plt  # для визуализации
from sklearn.linear_model import LinearRegression  # модель линейной регрессии

# Создание синтетической выборки данных
np.random.seed(42)  # фиксируем random seed для воспроизводимости
N = 10  # количество точек в обучающей выборке
X = np.linspace(-5, 5, N).reshape(-1, 1)  # создаем N точек от -5 до 5
y = np.sin(X) + np.random.normal(0, 0.2, size=N).reshape(-1, 1)  # целевая переменная: синус + шум

# Создание тестовой выборки (в 2 раза меньше)
X_test = np.linspace(-5, 5, N // 2).reshape(-1, 1)
y_test = np.sin(X_test) + np.random.normal(0, 0.2, size=N // 2).reshape(-1, 1)

# Очень простая модель (регрессия на константу) - просто среднее значение y
too_simple_model_predictions = np.mean(y) * np.ones_like(y)

# В меру сложная модель (полином 3-й степени)
X_ok = np.hstack([X, X ** 2, X ** 3])  # добавляем квадратичный и кубический признаки
ok_model = LinearRegression()  # создаем модель
ok_model.fit(X_ok, y)  # обучаем модель
ok_model_predictions = ok_model.predict(X_ok)  # получаем предсказания

# Очень сложная модель (полином 10-й степени) - может переобучаться
X_compl = np.hstack([X, X ** 2, X ** 3, X ** 4, X ** 5, 
                    X ** 6, X ** 7, X ** 8, X ** 9, X ** 10])  # много признаков
compl_model = LinearRegression()  # создаем модель
compl_model.fit(X_compl, y)  # обучаем модель
compl_model_predictions = compl_model.predict(X_compl)  # получаем предсказания

# Визуализация результатов
plt.figure(figsize=(10, 7))  # создаем фигуру

# Рисуем точки обучающей и тестовой выборок
plt.scatter(X, y, label='Тренировочная выборка')
plt.scatter(X_test, y_test, c='r', label='Тестовая выборка')

# Рисуем предсказания всех трех моделей
plt.plot(X, too_simple_model_predictions, label='Очень простая модель')
plt.plot(X, ok_model_predictions, label='В меру сложная модель')
plt.plot(X, compl_model_predictions, label='Очень сложная модель')

plt.grid()  # добавляем сетку
plt.legend()  # добавляем легенду

- Очень простая модель имеет большое смещение (bias), но малую (нулевую) дисперсию (variance). Модель явно недообучена.

- В меру сложная модель имеет небольшое смещение (bias) и небольшую дисперсию (variance).

- Очень сложная модель имеет небольшое смещение (bias), но большую дисперсию (variance). Модель явно переобучена.

**Задание:** пользуясь определениями выше, объясните, почему это так.

**Задание:** прокомментируйте величину смещения и дисперсии для следующих моделей:

1. Линейная регрессия, обучаемая на большой выборке без выбросов и линейно зависимых признаков, в которой признаки сильно коррелируют с целевой переменной.
2. Решающее дерево, которое строится до тех пор, пока в листах не окажется по одному объекту.
3. Логистическая регрессия, относящая все точки к одному классу.

### 1.A. Bias-Variance tradeoff

Из описания выше можно заметить, что при обучении моделей возникает выбор между смещением и дисперсией: недообученная модель имеет низкую дисперсию, но высокое смещение, а переобученная – низкое смещение, но высокую дисперсию. Этот выбор можно отобразить на картинке ([источник](https://www.bradyneal.com/bias-variance-tradeoff-textbooks-update)). Здесь Total Error – ошибка на тестовой выборке (generalization error).

![](https://www.bradyneal.com/img/bias-variance/fortmann-roe-bias-variance.png)

Вывод из неё очевиден: строить следует оптимальные по сложности модели. 

Возникает ли такой выбор при обучении любой модели? Последние исследования показывают, что поведение ошибки при обучении некоторых (современных) моделей не соответствует такой U-образной форме. Например, было показано, что ошибка на тестовой выборке продолжает убывать при расширении (увеличении числа слоёв) нейронных сетей:

<img src="https://www.bradyneal.com/img/bias-variance/neyshabur.jpg" alt="drawing" width="400"/>

В таких моделях поведение ошибки приобретает сложный вид:


<img src="https://www.bradyneal.com/img/bias-variance/double_descent.jpg" alt="drawing" width="800"/>

### 3. От деревьев к случайному лесу

#### 3.1 Решающее дерево

Мотивацию построения алгоритма случайного леса (Random Forest) удобно рассматривать в терминах смещения и дисперсии. Начнём с построения решающего дерева.

In [ ]:
# Пример отсюда: http://rasbt.github.io/mlxtend/user_guide/evaluate/bias_variance_decomp/

# Импорт необходимых модулей
from sklearn.model_selection import train_test_split  # для разделения данных на train/test
from mlxtend.data import boston_housing_data  # для загрузки датасета Boston Housing

# Загрузка данных
# X - матрица признаков (характеристики домов)
# y - вектор целевой переменной (цены домов)
X, y = boston_housing_data()

# Разделение данных на обучающую и тестовую выборки
# Параметры:
# test_size=0.3 - 30% данных пойдут в тестовую выборку
# random_state=123 - фиксируем seed для воспроизводимости результатов
# shuffle=True - перемешиваем данные перед разделением (важно для корректного обучения)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                   test_size=0.3,
                                                   random_state=123,
                                                   shuffle=True)

In [ ]:
# TODO: обучите решающее дерево без ограничений на тренировочной выборке
# TODO: рассчитайте MSE на тренировочной и тестовой выборках

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# 1. Создаем и обучаем решающее дерево БЕЗ ограничений
# max_depth=None - нет ограничения на глубину
# min_samples_split=2 - минимальное число образцов для разделения узла
# random_state=123 для воспроизводимости

dtree = DecisionTreeRegressor(max_depth=None, min_samples_split=2, random_state=123)
dtree.fit(X_train, y_train)  # обучение модели

# 2. Делаем предсказания на обеих выборках
y_train_pred = dtree.predict(X_train)  # предсказания на тренировочных данных
y_test_pred  = dtree.predict(X_test)    # предсказания на тестовых данных

# 3. Рассчитываем MSE (Mean Squared Error) для обеих выборок
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse  = mean_squared_error(y_test, y_test_pred)

# Выводим результаты
print(f"MSE на тренировочной выборке: {train_mse:.4f}")
print(f"MSE на тестовой выборке: {test_mse:.4f}")

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

# TODO: воспользуйтесь функцией bias_variance_decomp и выведите среднее смещение и среднюю дисперсию модели
# на тестовой выборке

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    estimator=DecisionTreeRegressor(max_depth=None, min_samples_split=2, random_state=123),
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    loss='mse',
    num_rounds=300,
    random_seed=123
)

# Вывод результатов
print(f"Средняя ожидаемая ошибка (MSE): {avg_expected_loss:.4f}")
print(f"Среднее смещение (Bias²): {avg_bias:.4f}") 
print(f"Средняя дисперсия (Variance): {avg_var:.4f}")

На лекции мы обсуждали, что один из способов борьбы с переобучением – построение композиций моделей. На этом семинаре мы рассмотрим построение композиций при помощи бэггинга.

#### 3.2 Бэггинг

Вспомним суть алгоритма:

1. Обучаем много деревьев на бутстрапированных подвыборках исходной выборки независимо друг от друга. Бутстрапированную подвыборку строим при помощи выбора $N$ (размер исходной выборки) наблюдений из исходной выборки с возвращением. 

2. Усредняем предсказания всех моделей (например, берём арифметическое среднее). 

Можно показать, что модель, построенная при помощи бэггинга, будет иметь **то же смещение**, что и у отдельных деревьев, но значительно **меньшую дисперсию** (при выполнении некоторых условий). 

In [ ]:
from sklearn.ensemble import BaggingRegressor

base_tree = DecisionTreeRegressor(random_state = 123)

# TODO: обучите бэггинг с 20 деревьями, каждое из которых строится без ограничений
# TODO: выведите среднее смещение и среднюю дисперсию модели на тестовой выборке

# Создаем модель BaggingRegressor
# base_estimator=base_tree - базовый алгоритм (дерево решений)
# n_estimators=20 - количество деревьев в ансамбле
# random_state=123 - фиксируем seed для воспроизводимости результатов

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    estimator=BaggingRegressor(estimator=base_tree, n_estimators=20, random_state=123),
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    loss='mse',
    num_rounds=100,
    random_seed=123
)

# Вывод результатов
print(f"Средняя ожидаемая ошибка (MSE): {avg_expected_loss:.4f}")
print(f"Среднее смещение (Bias²): {avg_bias:.4f}") 
print(f"Средняя дисперсия (Variance): {avg_var:.4f}")

Как мы видим, по сравнению с единичным деревом смещение практически не изменилось, но дисперсия уменьшилась в несколько раз!

Посмотрим, как это отразилось на среднеквадратичной ошибке.

In [ ]:
# TODO: рассчитайте MSE на тренировочной и тестовой выборках для бэггинга

bagging = BaggingRegressor(estimator=base_tree, n_estimators=20, random_state=123)
bagging.fit(X_train, y_train)  # обучение модели

# 2. Делаем предсказания на обеих выборках
y_train_pred = bagging.predict(X_train)  # предсказания на тренировочных данных
y_test_pred  = bagging.predict(X_test)    # предсказания на тестовых данных

# 3. Рассчитываем MSE (Mean Squared Error) для обеих выборок
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse  = mean_squared_error(y_test, y_test_pred)

# Выводим результаты
print(f"MSE на тренировочной выборке: {train_mse:.4f}")
print(f"MSE на тестовой выборке: {test_mse:.4f}")

Среднеквадратичная ошибка на тренировочной выборке больше не равна 0, а на тестовой – уменьшилась, что говорит о том, что мы успешно победили переобучение единичного решающего дерева. 

Можем ли мы снизить переобучение ещё сильнее? Можем!

#### 3.3 Случайный лес

При построении каждого дерева в бэггинге в ходе создания очередного узла будем выбирать случайный набор признаков, на основе которых производится разбиение. В результате такой процедуры мы уменьшим корреляцию между деревьями, за счёт чего снизим дисперсию итоговой модели. Такой алгоритм назвывается **случайным лесом** (Random Forest). 

По сравнению с единичным деревом к параметрам случайного леса добавляются:
- `max_features` – число признаков, на основе которых проводятся разбиения при построении дерева.

- `n_estimators` – число деревьев. 

Естественно, все параметры, относящиеся к единичному дереву, сохраняются для случайного леса.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: обучите случайный лес с 20 деревьями, каждое из которых строится без ограничений
# TODO: выведите среднее смещение и среднюю дисперсию модели на тестовой выборке
# TODO: рассчитайте MSE на тренировочной и тестовой выборках для случайного леса


avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    estimator=RandomForestRegressor(n_estimators=20, random_state=123),
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    loss='mse',
    num_rounds=100,
    random_seed=123
)

# Вывод результатов
print(f"Средняя ожидаемая ошибка (MSE): {avg_expected_loss:.4f}")
print(f"Среднее смещение (Bias²): {avg_bias:.4f}") 
print(f"Средняя дисперсия (Variance): {avg_var:.4f}")

rand_forest = RandomForestRegressor(n_estimators=20, random_state=123)
rand_forest.fit(X_train, y_train)  # обучение модели

# 2. Делаем предсказания на обеих выборках
y_train_pred = rand_forest.predict(X_train)  # предсказания на тренировочных данных
y_test_pred  = rand_forest.predict(X_test)    # предсказания на тестовых данных

# 3. Рассчитываем MSE (Mean Squared Error) для обеих выборок
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse  = mean_squared_error(y_test, y_test_pred)

# Выводим результаты
print(f"MSE на тренировочной выборке: {train_mse:.4f}")
print(f"MSE на тестовой выборке: {test_mse:.4f}")

Как мы видим, по сравнению с бэггингом смещение вновь осталось практически неизменным, а дисперсия немного уменьшилась. Конечно, если подобрать хорошие гиперпараметры, то получится снизить дисперсию ещё больше. 

Ошибка на тренировочной выборке увеличилась, а на тестовой – уменьшилась, что означает, что мы добились нашей цели в борьбе с переобученными деревьями!

### 4. Особенности случайного леса

#### 4.1 Число деревьев и "Случайный лес не переобучается"

В своём [блоге](https://www.stat.berkeley.edu/~breiman/RandomForests/cc_home.htm#remarks) Лео Бриман (Leo Breiman), создатель случайного леса, написал следующее:

> Random forest does not overfit. You can run as many trees as you want.

**Обратите внимание:** как говорилось на лекции, случайный лес не переобучается именно с ростом числа деревьев (за счёт совместной работы бэггинга и использования случайных подпространств), но не в принципе. Посмотрим на поведение случайного леса при росте числа деревьев.

In [ ]:
X, y = boston_housing_data()
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 123,
                                                    shuffle = True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 123)

In [ ]:
n_trees = 100
train_loss = []
test_loss = []

for i in range(1, n_trees):
    rf = RandomForestRegressor(n_estimators = i, random_state = 123)
    rf.fit(X_train, y_train)
    train_loss.append(mean_squared_error(y_train, rf.predict(X_train)))
    test_loss.append(mean_squared_error(y_test, rf.predict(X_test)))
    
plt.figure(figsize = (10, 7))
plt.grid()
plt.plot(train_loss, label = 'MSE_train')
plt.plot(test_loss, label = 'MSE_test')
plt.ylabel('MSE')
plt.xlabel('# trees')
plt.legend();

Как и ожидалось, по достижении некоторого числа деревьев обе ошибки практически не изменяются, то есть переобучения при росте числа деревьев не происходит.

Однако практика показывает, что при изменении какого-нибудь другого параметра на реальных данных переобучение может произойти: [пример 1](https://datascience.stackexchange.com/questions/1028/do-random-forest-overfit), [пример 2](https://mljar.com/blog/random-forest-overfitting/). Например, случайный лес с ограниченными по глубине деревьями может предсказывать более точно, чем лес без ограничений. 

В нашем же случае случайный лес, скорее, лишь страдает от регуляризации. Например, посмотрим на поведение модели при изменении максимальной глубины деревьев (поэксперементируйте с другими параметрами).

In [ ]:
max_depth = 40
train_loss = []
test_loss = []

for i in range(1, max_depth):
    rf = RandomForestRegressor(n_estimators = 20, max_depth = i, random_state = 123)
    rf.fit(X_train, y_train)
    train_loss.append(mean_squared_error(y_train, rf.predict(X_train)))
    test_loss.append(mean_squared_error(y_test, rf.predict(X_test)))
    
plt.figure(figsize = (10, 7))
plt.grid()
plt.plot(train_loss, label = 'MSE_train')
plt.plot(test_loss, label = 'MSE_test')
plt.ylabel('MSE')
plt.xlabel('max_depth')
plt.legend();

Переобучение не наблюдается. Вообще же, как обычно, гиперпараметры случайного леса стоит подбирать на кросс-валидации.

#### 4.2 Out-of-bag-ошибка

Как мы обсудили выше, при построении случайного леса каждое дерево строится на бутстрапированной подвыборке, полученной из исходной обучающей выборки случайным набором с повторениями. Понятно, что некоторые наблюдения попадут в такую подвыборку несколько раз, а некоторые не войдут в неё вообще. Для каждого дерева мы можем рассмотреть объекты, которые не участвовали в обучении и использовать их для валидации.

Усреднённая ошибка на неотобранных образцах по всему случайному лесу называется **out-of-bag-ошибкой**.

In [ ]:
X, y = boston_housing_data()
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 123,
                                                    shuffle = True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 123)

# oob_score_ = R2 на невиденных наблюдениях.
rf = RandomForestRegressor(n_estimators = 100, random_state = 123, oob_score = True)
rf.fit(X_train, y_train)
rf.oob_score_

#### 4.3 Важность признаков

Как и решающие деревья, случайный лес позволяет оценивать важность признаков.

In [ ]:
# Просто чтобы подгрузить названия признаков
from sklearn.datasets import load_boston
data = load_boston()

In [ ]:
plt.figure(figsize = (10, 7))
plt.bar(np.arange(len(rf.feature_importances_)), rf.feature_importances_);

Будьте осторожны с сильно коррелирующими признаками. Посмотрим, что произойдёт с важностью, если добавить в выборку линейно зависимый признак.

In [ ]:
RM_mc = (X_train[:, 5] * 2 + 3).reshape(-1, 1)
X_train_new = np.hstack((X_train, RM_mc))

In [ ]:
rf.fit(X_train_new, y_train)

In [ ]:
plt.figure(figsize = (10, 7))
names = list(np.arange(len(rf.feature_importances_) - 1).astype(str))
names.append('RM_mc')
plt.bar(names, rf.feature_importances_);

Важности перераспределились между линейной зависимыми признаками `RM` и `RM_mc`. Не забывайте учитывать корреляции между признаками, если вы используете этот метод для отбора признаков. Также обратите внимание на предупреждение в документации `sklearn`: не стоит использовать этот метод и для признаков, в которых есть много уникальных значений (например, категориальные признаки с небольшим числом категорий). 

### 5. Тестирование случайного леса на разных данных

Ниже представлены шаблоны для сравнения случайного леса и других моделей на данных разных типов. Проведите побольше экспериментов, используя разные модели и метрики. Попробуйте подобрать гиперпараметры случайного леса так, чтобы достичь какого-нибудь порога качества. 

**Внимание:** в этой части вам предстоит скачивать объёмные наборы данных. Не забудьте удалить их после семинара, если не планируете использовать их в дальнейшем, чтобы они не занимали лишнее место на вашем компьютере.

**! Случайный лес может обучаться достаточно долго.**

#### 5.1 Бинарная классификация на примере [Kaggle Predicting a Biological Response](https://www.kaggle.com/c/bioresponse/data?select=train.csv)

In [ ]:
# Загрузка данных
!wget  -O 'kaggle_response.csv' -q 'https://www.dropbox.com/s/uha70sej5ugcrur/_train_sem09.csv?dl=1'

In [ ]:
data = pd.read_csv('kaggle_response.csv')
X = data.iloc[:, 1:].values
y = data.iloc[:, 0].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 123)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# TODO: обучите логистическую регрессию и случайный лес с дефолтными параметрами
# Сравните их AUC ROC на тестовой выборке

# Обучение логистической регрессии
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict_proba(X_test)[:, 1] # вероятности принадлежности к классу 1
logreg_auc = roc_auc_score(y_test, logreg_pred)

# Обучение случайного леса
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)

# Вывод результатов
print(f"AUC ROC для логистической регрессии: {logreg_auc:.4f}")
print(f"AUC ROC для случайного леса: {rf_auc:.4f}")

#### 5.2 Изображения на примере [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist)

In [ ]:
# Загрузка данных
import torchvision

fmnist = torchvision.datasets.FashionMNIST('./', download = True)
X = fmnist.data.numpy().reshape(-1, 28 * 28)
y = fmnist.targets.numpy()

In [ ]:
plt.imshow(X[0, :].reshape(28, 28))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 123)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# TODO: обучите случайный лес и kNN с дефолтными параметрами
# Сравните их доли правильных ответов на тестовой выборке

# Обучение kNN
knn = KNeighborsClassifier()  # Используем параметры по умолчанию (n_neighbors=5)
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)

# Обучение случайного леса
rf = RandomForestClassifier()  # Используем параметры по умолчанию
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

# Вывод результатов
print(f"Доля правильных ответов для kNN: {knn_accuracy:.4f}")
print(f"Доля правильных ответов для случайного леса: {rf_accuracy:.4f}")

#### 5.3 Тексты на примере бинарной классификации твитов 

Скачиваем куски датасета ([источник](http://study.mokoron.com/)): [положительные](https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv?dl=0), [отрицательные](https://www.dropbox.com/s/r6u59ljhhjdg6j0/negative.csv).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MaxAbsScaler

# Предобработка из семинара 10
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = 'positive'
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = 'negative'
df = positive.append(negative)

X_train, X_test, y_train, y_test = train_test_split(df.text, df.label, random_state=13)

vec = CountVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(X_train)
bow_test = vec.transform(X_test)

scaler = MaxAbsScaler()
bow = scaler.fit_transform(bow)
bow_test = scaler.transform(bow_test)

X_train = bow
X_test = bow_test

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: обучите случайный лес с числом деревьев 100 и макс. глубиной дерева 20 
# и решающее дерево с макс. глубиной 20
# Сравните их доли правильных ответов на тестовой выборке

# Обучение решающего дерева
dt = DecisionTreeClassifier(max_depth=20, random_state=13)  # Фиксируем random_state для воспроизводимости
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

# Обучение случайного леса
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=13)  # Фиксируем random_state для воспроизводимости
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

# Вывод результатов
print(f"Доля правильных ответов для решающего дерева: {dt_accuracy:.4f}")
print(f"Доля правильных ответов для случайного леса: {rf_accuracy:.4f}")